In [5]:
UNI_RANDOM_SEED = 2024

import numpy as np
import torch
import torch.nn.functional as F

np.random.seed(UNI_RANDOM_SEED) 
torch.manual_seed(UNI_RANDOM_SEED)

torch.cuda.manual_seed(UNI_RANDOM_SEED)
torch.cuda.manual_seed_all(UNI_RANDOM_SEED)

import pdb
from pathlib import Path

try:
    import open3d
    from visual_utils import open3d_vis_utils as V
    OPEN3D_FLAG = True
except:
    import mayavi.mlab as mlab
    from visual_utils import visualize_utils as V
    OPEN3D_FLAG = False

from pcdet.datasets.kitti.kitti_dataset import create_kitti_infos
from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.datasets import KittiDataset, build_dataloader
from pcdet.models import build_network, load_data_to_gpu
from pcdet.utils import common_utils

from eval_utils import eval_utils


EVAL_OUTPUT_DIR = "./eval_output/"
CFG_FILE = "./cfgs/kitti_models/pointrcnn.yaml"
DATA_CONFIG_FILE = "./cfgs/dataset_configs/kitti_dataset.yaml"
DATA_PATH = "/home/ksas/Public/datasets/KITTI"
CKPT_PATH = "/home/ksas/Public/model_zoo/pcdet/pointrcnn_7870.pth"

BATCH_SIZE = 1
WORKERS = 4
DIST_TEST = False

cfg_from_yaml_file(CFG_FILE, cfg)

# BATCH_SIZE = cfg.OPTIMIZATION.BATCH_SIZE_PER_GPU
logger = common_utils.create_logger()
logger.info('-----------------Kitti Attack Test-------------------------')


2024-01-10 14:08:23,418   INFO  -----------------Kitti Attack Test-------------------------
2024-01-10 14:08:23,418   INFO  -----------------Kitti Attack Test-------------------------


In [6]:
test_set, test_loader, sampler = build_dataloader(
        dataset_cfg=cfg.DATA_CONFIG,
        class_names=cfg.CLASS_NAMES,
        batch_size=BATCH_SIZE,
        dist=DIST_TEST, workers=WORKERS, logger=logger, training=False
    )
logger.info(f'Class names of samples: \t{test_set.class_names}')

2024-01-10 14:08:23,429   INFO  Loading KITTI dataset
2024-01-10 14:08:23,429   INFO  Loading KITTI dataset
2024-01-10 14:08:23,582   INFO  Total samples for KITTI dataset: 3769
2024-01-10 14:08:23,582   INFO  Total samples for KITTI dataset: 3769
2024-01-10 14:08:23,584   INFO  Class names of samples: 	['Car', 'Pedestrian', 'Cyclist']
2024-01-10 14:08:23,584   INFO  Class names of samples: 	['Car', 'Pedestrian', 'Cyclist']


In [7]:
model = build_network(model_cfg=cfg.MODEL, num_class=len(cfg.CLASS_NAMES), dataset=test_set)
model.load_params_from_file(filename=CKPT_PATH, logger=logger, to_cpu=True)
model.cuda()
model.eval()

for idx, module in enumerate(model.module_list):
    logger.info(f'Module names of model \t({idx}): \t{module._get_name()}')
    
backbone_network = model.module_list[0]
point_headbox = model.module_list[1]
pointrcnn_head = model.module_list[2]

2024-01-10 14:08:23,684   INFO  ==> Loading parameters from checkpoint /home/ksas/Public/model_zoo/pcdet/pointrcnn_7870.pth to CPU
2024-01-10 14:08:23,684   INFO  ==> Loading parameters from checkpoint /home/ksas/Public/model_zoo/pcdet/pointrcnn_7870.pth to CPU
2024-01-10 14:08:23,784   INFO  ==> Done (loaded 309/309)
2024-01-10 14:08:23,784   INFO  ==> Done (loaded 309/309)
2024-01-10 14:08:23,796   INFO  Module names of model 	(0): 	PointNet2MSG
2024-01-10 14:08:23,796   INFO  Module names of model 	(0): 	PointNet2MSG
2024-01-10 14:08:23,797   INFO  Module names of model 	(1): 	PointHeadBox
2024-01-10 14:08:23,797   INFO  Module names of model 	(1): 	PointHeadBox
2024-01-10 14:08:23,798   INFO  Module names of model 	(2): 	PointRCNNHead
2024-01-10 14:08:23,798   INFO  Module names of model 	(2): 	PointRCNNHead


In [8]:
eval_utils.eval_one_epoch(
        cfg, None, model, test_loader, 0, logger, dist_test=DIST_TEST,
        result_dir=Path(EVAL_OUTPUT_DIR)
        , infer_time=True
    )

2024-01-10 14:08:23,809   INFO  *************** EPOCH 0 EVALUATION *****************
2024-01-10 14:08:23,809   INFO  *************** EPOCH 0 EVALUATION *****************
eval: 100%|██████████| 3769/3769 [03:06<00:00, 20.18it/s, infer_time=50.83(46.32), recall_0.3=(15815, 15831) / 17558]
2024-01-10 14:11:30,614   INFO  *************** Performance of EPOCH 0 *****************
2024-01-10 14:11:30,614   INFO  *************** Performance of EPOCH 0 *****************
2024-01-10 14:11:30,615   INFO  Generate label finished(sec_per_example: 0.0001 second).
2024-01-10 14:11:30,615   INFO  Generate label finished(sec_per_example: 0.0001 second).
2024-01-10 14:11:30,616   INFO  recall_roi_0.3: 0.900729
2024-01-10 14:11:30,616   INFO  recall_roi_0.3: 0.900729
2024-01-10 14:11:30,616   INFO  recall_rcnn_0.3: 0.901640
2024-01-10 14:11:30,616   INFO  recall_rcnn_0.3: 0.901640
2024-01-10 14:11:30,617   INFO  recall_roi_0.5: 0.865588
2024-01-10 14:11:30,617   INFO  recall_roi_0.5: 0.865588
2024-01-10 1

{'recall/roi_0.3': 0.9007290124159927,
 'recall/rcnn_0.3': 0.9016402779359836,
 'recall/roi_0.5': 0.8655883358013441,
 'recall/rcnn_0.5': 0.8726506435812735,
 'recall/roi_0.7': 0.6834491399931655,
 'recall/rcnn_0.7': 0.7353912746326461,
 'Car_aos/easy_R40': 96.43589171706566,
 'Car_aos/moderate_R40': 92.84116830033655,
 'Car_aos/hard_R40': 90.40826808758254,
 'Car_3d/easy_R40': 91.7462721215099,
 'Car_3d/moderate_R40': 80.62902917926857,
 'Car_3d/hard_R40': 78.15952894787168,
 'Car_bev/easy_R40': 93.26745215876231,
 'Car_bev/moderate_R40': 89.14557982657577,
 'Car_bev/hard_R40': 86.89832173323924,
 'Car_image/easy_R40': 96.4562091272675,
 'Car_image/moderate_R40': 92.95102763439367,
 'Car_image/hard_R40': 90.56066701243235,
 'Pedestrian_aos/easy_R40': 73.14217132584514,
 'Pedestrian_aos/moderate_R40': 66.50788122993319,
 'Pedestrian_aos/hard_R40': 59.8830884977154,
 'Pedestrian_3d/easy_R40': 62.79258976139079,
 'Pedestrian_3d/moderate_R40': 54.99338364578974,
 'Pedestrian_3d/hard_R40':